---
## Step 0 — Imports and settings

In [5]:
%load_ext autoreload
%autoreload 2
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np

sc.settings.verbosity = 2  # show INFO messages from scanpy
sc.settings.set_figure_params(dpi=100, facecolor="white")

print(f"scanpy  version: {sc.__version__}")
print(f"anndata version: {ad.__version__}")
print(f"pandas  version: {pd.__version__}")
print(f"numpy   version: {np.__version__}")

from besca_sc.io import load_cellbender
from besca_sc.paths import (
    EXTERNAL_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    RAW_DIR,
    RESULTS_DIR,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
scanpy  version: 1.12.4
anndata version: 0.13.4
pandas  version: 3.0.6
numpy   version: 2.5.4


/var/folders/22/sjkvvcj17_gdnvz15k1nj1800000gn/T/ipykernel_86441/3423734026.py:9: FutureWarning: The method set_figure_params is deprecated and will be removed in the future. Use :func:`scanpy.set_figure_params` instead
  sc.settings.set_figure_params(dpi=100, facecolor="white")
/var/folders/22/sjkvvcj17_gdnvz15k1nj1800000gn/T/ipykernel_86441/3423734026.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"scanpy  version: {sc.__version__}")
/var/folders/22/sjkvvcj17_gdnvz15k1nj1800000gn/T/ipykernel_86441/3423734026.py:12: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f"anndata version: {ad.__version__}")


---
## Step 1 — Define file paths

All paths are absolute so the notebook works from any working directory.

In [6]:
# CellBender and HADGE files are resolved from the central project paths.
CB_INF = RAW_DIR / "Infected" / "cellbender_filtered.h5"
CB_UNINF = RAW_DIR / "Uninfected" / "cellbender_filtered.h5"

# HADGE consensus demultiplexing results (7-tool majority vote)
# Columns: Barcode, [7 tool assignments], Consensus_Assignment, Confidence,
#          Consensus_DropletType, Donor, Condition
DEMUX_INF = RAW_DIR / "Infected" / "Infected.consensus.csv"
DEMUX_UNINF = RAW_DIR / "Uninfected" / "Uninfected.consensus.csv"

# Verify all input files exist before proceeding
for f in [CB_INF, CB_UNINF, DEMUX_INF, DEMUX_UNINF]:
    status = "OK" if f.exists() else "MISSING!"
    print(f"  [{status}]  {f}")

  [OK]  /Users/adaml9/besca_sc/data/raw/Infected/cellbender_filtered.h5
  [OK]  /Users/adaml9/besca_sc/data/raw/Uninfected/cellbender_filtered.h5
  [OK]  /Users/adaml9/besca_sc/data/raw/Infected/Infected.consensus.csv
  [OK]  /Users/adaml9/besca_sc/data/raw/Uninfected/Uninfected.consensus.csv


---
## Step 2 — Load CellBender filtered h5 files

`cellbender_filtered.h5` is in standard 10x HDF5 format.  
It contains **both** Gene Expression counts and Antibody Capture (HTO) counts.  
We keep only the Gene Expression features here; the HTO assignment is already handled by HADGE.

In [7]:
print("Loading CellBender filtered matrices ...")
adata_inf = load_cellbender("Infected")
adata_uninf = load_cellbender("Uninfected")

Loading CellBender filtered matrices ...
reading /Users/adaml9/besca_sc/data/raw/Infected/cellbender_filtered.h5


 (0:00:02)


/Users/adaml9/besca_sc/.pixi/envs/besca/lib/python3.12/site-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


reading /Users/adaml9/besca_sc/data/raw/Uninfected/cellbender_filtered.h5
 (0:00:03)


/Users/adaml9/besca_sc/.pixi/envs/besca/lib/python3.12/site-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


---
## Step 3 — Load HADGE consensus demultiplexing results

The consensus CSV has one row per cell barcode with:
- Individual calls from 7 HTO tools
- `Consensus_Assignment` — majority-vote hashtag
- `Consensus_DropletType` — `singlet`, `doublet`, or `negative`
- `Donor` — `P522` or `P145` (populated only for singlets)
- `Condition` — `mono` or `co` (populated only for singlets)

In [8]:
def load_demux(path):
    """Load HADGE consensus CSV with Barcode as the row index."""
    df = pd.read_csv(path, index_col=0)  # first column (Barcode) becomes the index
    # Rename 'Condition' to 'stimulation' to avoid ambiguity
    df = df.rename(columns={"Condition": "stimulation"})
    return df


print("Loading HADGE consensus files ...")
demux_inf = load_demux(DEMUX_INF)
demux_uninf = load_demux(DEMUX_UNINF)

print(f"  Infected:   {len(demux_inf)} barcodes in demux table")
print(f"  Uninfected: {len(demux_uninf)} barcodes in demux table")

# Quick look at the table structure
demux_inf.head(3)

Loading HADGE consensus files ...
  Infected:   32701 barcodes in demux table
  Uninfected: 17646 barcodes in demux table


,htodemux_Assignment,multiseq_Assignment,hasheddrops_Assignment,gmmdemux_Assignment,hashsolo_Assignment,bff_consensuscall_Assignment,demuxmix_Assignment,Consensus_Assignment,Confidence,Consensus_DropletType,Donor,stimulation
Barcode,,,,,,,,,,,,
AAACCCAAGATCGACG-1,negative,negative,negative,negative,HuHashtag6,negative,negative,negative,6/7,negative,NaN,NaN
AAACCCAAGCCATGCC-1,negative,negative,negative,negative,HuHashtag6,negative,negative,negative,6/7,negative,NaN,NaN
AAACCCAAGCGTCGAA-1,negative,negative,negative,negative,HuHashtag6,negative,negative,negative,6/7,negative,NaN,NaN


---
## Step 4 — Add demultiplexing metadata to each AnnData object

Cell-level metadata lives in `adata.obs` (a pandas DataFrame with one row per cell).  
We join the HADGE table onto `adata.obs` matching on the barcode index.

In [9]:
def add_demux_metadata(adata, demux_df, library_name):
    """Join demux labels into adata.obs. Returns a copy with metadata added."""
    # Left-join: keep all cells from AnnData, add NaN where barcode not in demux table
    adata.obs = adata.obs.join(demux_df, how="left")

    # Report how many barcodes were successfully matched
    n_matched = adata.obs["Consensus_DropletType"].notna().sum()
    print(f"  {library_name}: {n_matched} / {adata.n_obs} cells matched to demux table")

    # Report droplet type breakdown
    print(f"  {library_name} droplet types:")
    print(adata.obs["Consensus_DropletType"].value_counts(dropna=False).to_string())
    print()
    return adata


print("Adding demux metadata ...")
adata_inf = add_demux_metadata(adata_inf, demux_inf, "Infected")
adata_uninf = add_demux_metadata(adata_uninf, demux_uninf, "Uninfected")

Adding demux metadata ...
  Infected: 32701 / 32701 cells matched to demux table
  Infected droplet types:
Consensus_DropletType
negative    17035
singlet     13649
doublet      2017

  Uninfected: 17646 / 17646 cells matched to demux table
  Uninfected droplet types:
Consensus_DropletType
singlet     15172
doublet      2395
negative       79



---
## Step 5 — Filter to singlets only

We keep only cells classified as `singlet` by the HADGE consensus.  
- **doublets** — two cells captured together (would confound downstream analysis)  
- **negative** — cells with no detectable hashtag signal (ambient or low HTO capture)

In [10]:
adata_inf_s = adata_inf[adata_inf.obs["Consensus_DropletType"] == "singlet"].copy()
adata_uninf_s = adata_uninf[
    adata_uninf.obs["Consensus_DropletType"] == "singlet"
].copy()

print(
    f"Infected   — singlets: {adata_inf_s.n_obs}  "
    f"(from {adata_inf.n_obs} total, {adata_inf_s.n_obs / adata_inf.n_obs * 100:.1f}%)"
)
print(
    f"Uninfected — singlets: {adata_uninf_s.n_obs}  "
    f"(from {adata_uninf.n_obs} total, {adata_uninf_s.n_obs / adata_uninf.n_obs * 100:.1f}%)"
)

Infected   — singlets: 13649  (from 32701 total, 41.7%)
Uninfected — singlets: 15172  (from 17646 total, 86.0%)


---
## Step 6 — Add library-level and sample-level metadata

Each cell gets:
- `library` — which sequencing pool it came from (`Infected` or `Uninfected`)
- `sample_id` — the full biological sample identity: `{library}_{Donor}_{stimulation}`  
  e.g. `Infected_P522_mono`, `Uninfected_P145_co`, …

In [11]:
def add_sample_metadata(adata, library_name):
    adata.obs["library"] = library_name
    adata.obs["sample_id"] = (
        adata.obs["library"]
        + "_"
        + adata.obs["Donor"].astype(str)
        + "_"
        + adata.obs["stimulation"].astype(str)
    )
    return adata


adata_inf_s = add_sample_metadata(adata_inf_s, "Infected")
adata_uninf_s = add_sample_metadata(adata_uninf_s, "Uninfected")

print("Infected — cells per sample:")
print(adata_inf_s.obs["sample_id"].value_counts().to_string())
print()
print("Uninfected — cells per sample:")
print(adata_uninf_s.obs["sample_id"].value_counts().to_string())

Infected — cells per sample:
sample_id
Infected_P145_co      4378
Infected_P145_mono    3416
Infected_P522_mono    2977
Infected_P522_co      2878

Uninfected — cells per sample:
sample_id
Uninfected_P522_co      4459
Uninfected_P145_mono    4310
Uninfected_P522_mono    3575
Uninfected_P145_co      2828


---
## Step 7 — Merge the two AnnData objects into one

We use `anndata.concat()` to combine the Infected and Uninfected datasets.  

Key parameters:
- `join='inner'` — keep only **genes present in both** datasets (avoids NaN genes)
- `index_unique='-'` — if the same barcode exists in both datasets, append `-Infected` / `-Uninfected` to make them unique

In [12]:
adata = ad.concat(
    {"Infected": adata_inf_s, "Uninfected": adata_uninf_s},
    label="library",  # adds / updates the 'library' column in adata.obs
    join="inner",  # keep genes present in BOTH datasets
    index_unique="-",  # makes barcodes unique: BARCODE-Infected / BARCODE-Uninfected
    merge="same",  # for var (gene) metadata: only keep columns identical in both
)

print(f"Merged AnnData: {adata.n_obs} cells  x  {adata.n_vars} genes")
print()
print("Cells per sample_id:")
print(adata.obs["sample_id"].value_counts().sort_index().to_string())

Merged AnnData: 28821 cells  x  38606 genes

Cells per sample_id:
sample_id
Infected_P145_co        4378
Infected_P145_mono      3416
Infected_P522_co        2878
Infected_P522_mono      2977
Uninfected_P145_co      2828
Uninfected_P145_mono    4310
Uninfected_P522_co      4459
Uninfected_P522_mono    3575


---
## Step 8 — Quick sanity check on the merged object

Before saving, verify that the metadata looks correct.

In [13]:
print("=== Merged AnnData summary ===")
print(adata)
print()
print("Metadata columns available in adata.obs:")
print(list(adata.obs.columns))
print()
print("First few rows of adata.obs:")
adata.obs[
    [
        "library",
        "Donor",
        "stimulation",
        "sample_id",
        "Consensus_DropletType",
        "Consensus_Assignment",
        "Confidence",
    ]
].head(6)

=== Merged AnnData summary ===
AnnData object with n_obs × n_vars = 28821 × 38606
    obs: 'htodemux_Assignment', 'multiseq_Assignment', 'hasheddrops_Assignment', 'gmmdemux_Assignment', 'hashsolo_Assignment', 'bff_consensuscall_Assignment', 'demuxmix_Assignment', 'Consensus_Assignment', 'Confidence', 'Consensus_DropletType', 'Donor', 'stimulation', 'library', 'sample_id'
    var: 'gene_ids', 'feature_types', 'genome'
    layers: None (.X)

Metadata columns available in adata.obs:
['htodemux_Assignment', 'multiseq_Assignment', 'hasheddrops_Assignment', 'gmmdemux_Assignment', 'hashsolo_Assignment', 'bff_consensuscall_Assignment', 'demuxmix_Assignment', 'Consensus_Assignment', 'Confidence', 'Consensus_DropletType', 'Donor', 'stimulation', 'library', 'sample_id']

First few rows of adata.obs:


,library,Donor,stimulation,sample_id,Consensus_DropletType,Consensus_Assignment,Confidence
CATGGTAGTATGAAAC-1-Infected,Infected,P145,co,Infected_P145_co,singlet,HuHashtag6,7/7
GCAACCGCAGCCCAGT-1-Infected,Infected,P145,co,Infected_P145_co,singlet,HuHashtag6,5/7
GCCTGTTAGCTAAATG-1-Infected,Infected,P522,mono,Infected_P522_mono,singlet,HuHashtag3,7/7
GTGTCCTGTGATACAA-1-Infected,Infected,P145,co,Infected_P145_co,singlet,HuHashtag6,7/7
GCTGAATAGTGCTAGG-1-Infected,Infected,P145,co,Infected_P145_co,singlet,HuHashtag6,7/7
CAATTTCGTCCAATCA-1-Infected,Infected,P522,mono,Infected_P522_mono,singlet,HuHashtag3,7/7


In [14]:
# Verify no NaN values in the key metadata columns for singlets
key_cols = ["library", "Donor", "stimulation", "sample_id", "Consensus_DropletType"]
nan_counts = adata.obs[key_cols].isna().sum()
print("NaN counts in key columns (should all be 0):")
print(nan_counts.to_string())

NaN counts in key columns (should all be 0):
library                  0
Donor                    0
stimulation              0
sample_id                0
Consensus_DropletType    0


---
## Step 9 — Save the merged AnnData object

`.h5ad` is the standard scanpy/AnnData file format. This is the starting point for all downstream QC and analysis.

In [16]:
out_path = f"{PROCESSED_DIR}/merged.h5ad"
adata.write_h5ad(out_path)
print(f"Saved to: {out_path}")

Saved to: /Users/adaml9/besca_sc/data/processed/merged.h5ad
